# VQE + Zero-Noise Extrapolation + Autodiff, on a Real Molecule

**For anyone new to this**: this notebook finds the ground-state (lowest) energy of a real molecule -- a chain of 10 hydrogen atoms -- using a technique called VQE (Variational Quantum Eigensolver). VQE works by running a small quantum circuit with adjustable knobs (parameters), measuring the energy it produces, and repeatedly nudging those knobs in the direction that lowers the energy -- the same idea as training a neural network, just applied to a quantum circuit instead. Getting the direction to nudge in (the *gradient*) automatically, instead of by trial and error, is what "autodiff" (automatic differentiation) does here, via [JAX](https://jax.readthedocs.io/).

The twist this notebook demonstrates: the *textbook* way to do this needs a matrix that grows so fast (4 to the power of the qubit count) that it becomes impossible to even store in memory well before you run out of qubits. [Dense-Evolution](https://github.com/tatopenn-cell/Dense-Evolution) avoids that matrix entirely (`PauliSumOperator`), while staying fully differentiable -- and combines it here with Zero-Noise Extrapolation (ZNE), a real technique for estimating what a *noisy* quantum computer would have measured if it had no noise at all.

**Full write-up with more detail**: https://tatopenn-cell.github.io/Dense-Evolution-Discovery/vqe_pauli_sum_zne_autodiff/

**References** (for going deeper):
- Peruzzo et al. 2014, ["A variational eigenvalue solver on a photonic quantum processor"](https://arxiv.org/abs/1304.3061) -- the original VQE paper.
- Temme, Bravyi & Gambetta 2017, ["Error Mitigation for Short-Depth Quantum Circuits"](https://arxiv.org/abs/1612.02058) -- the original Zero-Noise Extrapolation paper.
- Kandala et al. 2017, ["Hardware-efficient variational quantum eigensolver for small molecules and quantum magnets"](https://arxiv.org/abs/1704.05018) -- the ansatz style used below (alternating rotation + entangling layers).
- [PennyLane quantum chemistry docs](https://docs.pennylane.ai/en/stable/introduction/chemistry.html) -- the library used here to build the real molecular Hamiltonian.
- [Dense-Evolution on PyPI](https://pypi.org/project/dense-evolution/) / [documentation](https://tatopenn-cell.github.io/Dense-Evolution-Discovery/).

In [ ]:
!pip install -q dense-evolution pennylane optax

## 1. Build the real molecule's Hamiltonian

H10: a chain of 10 hydrogen atoms, 0.74 Angstrom apart, the standard VQE-scaling benchmark in the literature. At its full size this needs 20 qubits -- we use a reduced "active space" of 6 of its 10 orbitals (still the *same real molecule*, just fewer of its electrons/orbitals treated explicitly) to get 12 qubits, light enough to run comfortably here. The Hamiltonian comes back as a list of `(coefficient, {qubit: 'X'/'Y'/'Z'})` Pauli terms -- never a dense matrix.

In [ ]:
import numpy as np
import pennylane as qml


def linear_chain_geometry(n_atoms, bond_length_angstrom):
    """N atoms on a line, each bond_length_angstrom apart."""
    return np.array([[0.0, 0.0, i * bond_length_angstrom] for i in range(n_atoms)])


def pennylane_hamiltonian_to_pauli_terms(H):
    """Extracts (coeff, {qubit: 'X'|'Y'|'Z'}) terms from a PennyLane
    Hamiltonian -- the real Pauli decomposition PennyLane itself already
    computed internally, just handed over in the plain dict form
    dense_evolution accepts, instead of densifying it via qml.matrix()."""
    coeffs, ops = H.terms()
    terms = []
    for coeff, op in zip(coeffs, ops):
        pauli = {}
        factors = op.operands if hasattr(op, "operands") else [op]
        for factor in factors:
            wires = factor.wires
            if not len(wires) or factor.name == "Identity":
                continue
            pauli[int(wires[0])] = factor.name[-1]  # 'PauliZ' -> 'Z', etc.
        terms.append((float(np.real(complex(coeff))), pauli))
    return terms


geometry = linear_chain_geometry(10, 0.74)
molecule = qml.qchem.Molecule(["H"] * 10, geometry, charge=0, unit="angstrom")
H_pennylane, n_qubits = qml.qchem.molecular_hamiltonian(
    molecule, method="dhf", mapping="jordan_wigner",
    active_electrons=10, active_orbitals=6,
)
terms = pennylane_hamiltonian_to_pauli_terms(H_pennylane)
print(f"n_qubits = {n_qubits}, n_pauli_terms = {len(terms)}")

## 2. `PauliSumOperator`: the Hamiltonian without the matrix

First, a quick sanity check on a small system: `PauliSumOperator @ vector` must agree with the dense `Hamiltonian matrix @ vector`, since the whole point is that it's the same math, just computed without ever building the matrix.

In [ ]:
import jax
import jax.numpy as jnp
import dense_evolution as de
from dense_evolution.physics.observables import pauli_hamiltonian_to_matrix

de.set_precision(True)  # real chemistry needs float64, not JAX's float32 default

rng = np.random.default_rng(0)
small_terms = [(float(rng.normal()), "".join(rng.choice(list("IXYZ")) for _ in range(4))) for _ in range(10)]
h_dense = pauli_hamiltonian_to_matrix(small_terms, 4)
h_op = de.PauliSumOperator(small_terms, 4)
v = jnp.asarray(rng.normal(size=16) + 1j * rng.normal(size=16))
max_diff = float(jnp.max(jnp.abs((h_op @ v) - (h_dense @ np.asarray(v)))))
print(f"PauliSumOperator vs dense matrix, max_diff = {max_diff:.2e} (should be ~1e-15)")
assert max_diff < 1e-9

## 3. The ansatz circuit and the differentiable energy function

A "hardware-efficient" ansatz: layers of single-qubit RY rotations (the adjustable knobs) followed by a chain of CX gates to entangle the qubits, repeated twice. `circuit_to_energy_fn` turns this circuit into a JAX function `energy_fn(theta, h_matrix) -> (energy, statevector)` that `jax.grad` can differentiate -- and `h_matrix` here is our `PauliSumOperator`, not a dense matrix.

In [ ]:
def hardware_efficient_ansatz_qasm(n_qubits, n_layers):
    lines = ["OPENQASM 2.0;", 'include "qelib1.inc";', f"qreg q[{n_qubits}];"]
    for _ in range(n_layers):
        for q in range(n_qubits):
            lines.append(f"ry(0.0) q[{q}];")  # 0.0 is a placeholder; circuit_to_energy_fn injects theta
        for q in range(n_qubits - 1):
            lines.append(f"cx q[{q}],q[{q + 1}];")
    return "\n".join(lines)


qasm = hardware_efficient_ansatz_qasm(n_qubits, n_layers=2)
circuit = de.QASMParser().parse(qasm)
energy_fn, n_params = de.circuit_to_energy_fn(circuit, n_qubits)
h_op = de.PauliSumOperator(terms, n_qubits)
print(f"ansatz parameters: {n_params}")

## 4. Run VQE

The first call below compiles the whole energy+gradient computation into one fused kernel (all 919 Pauli terms fused into a single JAX-jit trace) -- this takes under a minute. Every call after that reuses the compiled kernel and is fast (a fraction of a second).

In [ ]:
import optax


@jax.jit
def loss(theta):
    energy, _ = energy_fn(theta, h_op)
    return energy


value_and_grad = jax.jit(jax.value_and_grad(loss))

key = jax.random.PRNGKey(0)
theta = 0.1 * jax.random.normal(key, (n_params,))
optimizer = optax.adam(0.1)
opt_state = optimizer.init(theta)

trace = []
for step in range(80):
    energy, grad = value_and_grad(theta)
    updates, opt_state = optimizer.update(grad, opt_state)
    theta = optax.apply_updates(theta, updates)
    trace.append(float(energy))
    if step % 20 == 0 or step == 79:
        print(f"step {step:3d}: E = {float(energy):.6f} Ha")

e_vqe = trace[-1]
print(f"\nE_vqe (converged) = {e_vqe:.6f} Ha")

## 5. Zero-Noise Extrapolation

Real quantum computers are noisy. ZNE estimates what a *noiseless* device would have measured by deliberately running the circuit at several *higher* noise levels, then extrapolating the trend back to zero noise. `dense_evolution`'s noise model draws one random error per qubit per call (a single "shot"), so we average many shots per noise level to get a stable estimate -- a single shot is too noisy on its own to extrapolate from.

In [ ]:
@jax.jit
def noisy_energy(theta, p, noise_key):
    noise = de.NoiseSpec(model="depolarizing", p=p, jax_key=noise_key)
    e, _ = energy_fn(theta, h_op, noise=noise)
    return e


noise_scales = [1.0, 2.0, 3.0]
base_p = 0.05
n_shots = 40

noisy_energies = []
for scale in noise_scales:
    p = base_p * scale
    shots = [float(noisy_energy(theta, p, jax.random.PRNGKey(int(scale * 100000) + s))) for s in range(n_shots)]
    mean_e = float(np.mean(shots))
    noisy_energies.append(mean_e)
    print(f"scale={scale:.1f}x (p={p:.3f}): E = {mean_e:.6f} +/- {float(np.std(shots)):.6f} Ha")

e_zne = float(de.zero_noise_extrapolation(jnp.array(noisy_energies), jnp.array(noise_scales)))
print(f"\nE_zne (extrapolated to zero noise) = {e_zne:.6f} Ha")
print(f"raw base-noise error = {abs(noisy_energies[0] - e_vqe):.6f} Ha")
print(f"ZNE-corrected error  = {abs(e_zne - e_vqe):.6f} Ha")

## 6. Plot the results

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))

ax1.plot(trace, color="#1f77b4", linewidth=1.6, label="VQE energy")
ax1.set_xlabel("Adam step")
ax1.set_ylabel("Energy (Ha)")
ax1.set_title(f"VQE convergence ({n_qubits} qubits)")
ax1.legend()
ax1.grid(alpha=0.25)

ax2.plot(noise_scales, noisy_energies, "o-", color="#d62728", label="noisy (measured)")
ax2.scatter([0.0], [e_zne], color="#9467bd", zorder=5, s=60, label=f"ZNE ({e_zne:.4f} Ha)")
ax2.axhline(e_vqe, color="#2ca02c", linestyle="--", linewidth=1.3, label=f"noiseless ({e_vqe:.4f} Ha)")
ax2.set_xlabel("Noise scale factor")
ax2.set_ylabel("Energy (Ha)")
ax2.set_title("Zero-Noise Extrapolation")
ax2.legend()
ax2.grid(alpha=0.25)

fig.tight_layout()
plt.show()

## Where to go next

- Try a different molecule: swap `["H"] * 10` and the geometry for any element PennyLane's STO-3G table covers (H through Ar).
- Try `active_orbitals=8` or higher for a bigger (16+ qubit) Hamiltonian -- watch the term count and compile time grow.
- Read the full write-up (with the real memory-wall numbers behind the 12-qubit choice, and a real bug this experiment caught and fixed along the way): https://tatopenn-cell.github.io/Dense-Evolution-Discovery/vqe_pauli_sum_zne_autodiff/
- Browse other experiments: https://tatopenn-cell.github.io/Dense-Evolution-Discovery/